In [2]:
# =========================================
# IMPORTS + DEVICE SETUP + RANDOM SEED
# =========================================

import os
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchaudio
import torchaudio.transforms as T
from torch.cuda.amp import autocast, GradScaler

from transformers import ASTForAudioClassification

import librosa
import librosa.display

from sklearn.metrics import f1_score, accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score

from tqdm import tqdm
import wandb

import warnings
warnings.filterwarnings("ignore")

2026-03-19 06:43:22.180816: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773902602.202180     211 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773902602.209019     211 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773902602.226100     211 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773902602.226117     211 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773902602.226121     211 computation_placer.cc:177] computation placer alr

In [3]:
# -----------------------------
# Device
# -----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [4]:
# -----------------------------
# Seed for reproducibility
# -----------------------------
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

In [5]:
# =========================================
# UPDATED CFG FOR AST v2
# =========================================

class CFG:
    
    DATA_ROOT = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup"
    GENRE_STEMS = os.path.join(DATA_ROOT, "genres_stems")
    MASHUP_DIR = os.path.join(DATA_ROOT, "mashups")
    TEST_CSV = os.path.join(DATA_ROOT, "test.csv")
    
    # Audio
    SAMPLE_RATE = 16000
    DURATION = 30
    TARGET_FRAMES = 1024
    MAX_LENGTH = SAMPLE_RATE * DURATION
    N_MELS = 128
    
    # Training
    BATCH_SIZE = 8
    EPOCHS = 25              # Increased
    LR = 5e-6                # Lower LR (more stable fine-tuning)
    WEIGHT_DECAY = 0.05      # Stronger regularization
    LABEL_SMOOTHING = 0.2    # Better generalization
    
    # Misc
    NUM_CLASSES = 10
    SEED = 42
    MODEL_SAVE_PATH = "ast_v4.pth"
    
    TRAIN_MODE = False

In [6]:
# =========================================
# LABEL MAPPING + TRAIN/VAL SPLIT
# =========================================

GENRES = [
    "blues", "classical", "country", "disco", "hiphop",
    "jazz", "metal", "pop", "reggae", "rock"
]

label2idx = {genre: idx for idx, genre in enumerate(GENRES)}
idx2label = {idx: genre for genre, idx in label2idx.items()}

print("Label mapping:", label2idx)


# Collect all song folders (each song contains 4 stems)
all_songs = []

for genre in GENRES:
    genre_path = os.path.join(CFG.GENRE_STEMS, genre)
    song_folders = os.listdir(genre_path)
    
    for song in song_folders:
        song_path = os.path.join(genre_path, song)
        all_songs.append((song_path, label2idx[genre]))

print("Total songs found:", len(all_songs))


# Train/Validation split
train_songs, val_songs = train_test_split(
    all_songs,
    test_size=0.2,
    stratify=[label for _, label in all_songs],
    random_state=CFG.SEED
)

print("Train samples:", len(train_songs))
print("Validation samples:", len(val_songs))

Label mapping: {'blues': 0, 'classical': 1, 'country': 2, 'disco': 3, 'hiphop': 4, 'jazz': 5, 'metal': 6, 'pop': 7, 'reggae': 8, 'rock': 9}
Total songs found: 1000
Train samples: 800
Validation samples: 200


In [7]:
# =========================================
# LOAD ESC-50 NOISE FILE PATHS
# =========================================

ESC_AUDIO_DIR = os.path.join(CFG.DATA_ROOT, "ESC-50-master", "audio")

noise_files = [
    os.path.join(ESC_AUDIO_DIR, f)
    for f in os.listdir(ESC_AUDIO_DIR)
    if f.endswith(".wav")
]

print("Total noise files found:", len(noise_files))

Total noise files found: 2000


In [8]:
# =========================
# TORCHAUDIO MASHUP DATASET 
# =========================

class MashupDataset(Dataset):
    def __init__(self, song_list, train=True):
        self.song_list = song_list
        self.train = train
        self.sample_rate = CFG.SAMPLE_RATE
        self.chunk_length = CFG.MAX_LENGTH
        self.target_frames = CFG.TARGET_FRAMES

        self.mel_transform = T.MelSpectrogram(
            sample_rate=self.sample_rate,
            n_fft=1024,
            hop_length=320,
            n_mels=CFG.N_MELS,
            f_max=8000
        )

        self.db_transform = T.AmplitudeToDB()

        self.time_mask = T.TimeMasking(time_mask_param=40)
        self.freq_mask = T.FrequencyMasking(freq_mask_param=20)

    def load_audio(self, path):
        waveform, sr = torchaudio.load(path)

        if sr != self.sample_rate:
            waveform = torchaudio.functional.resample(
                waveform, sr, self.sample_rate
            )

        waveform = waveform.mean(dim=0)  # mono

        if waveform.shape[0] < self.chunk_length:
            pad_len = self.chunk_length - waveform.shape[0]
            waveform = torch.nn.functional.pad(waveform, (0, pad_len))
            return waveform

        if self.train:
            start = random.randint(0, waveform.shape[0] - self.chunk_length)
        else:
            start = (waveform.shape[0] - self.chunk_length) // 2

        waveform = waveform[start:start+self.chunk_length]
        return waveform

    def add_noise(self, audio):
        noise_path = random.choice(noise_files)
        noise, sr = torchaudio.load(noise_path)

        if sr != self.sample_rate:
            noise = torchaudio.functional.resample(
                noise, sr, self.sample_rate
            )

        noise = noise.mean(dim=0)

        if noise.shape[0] < audio.shape[0]:
            pad_len = audio.shape[0] - noise.shape[0]
            noise = torch.nn.functional.pad(noise, (0, pad_len))
        else:
            noise = noise[:audio.shape[0]]

        signal_power = audio.pow(2).mean()
        noise_power = noise.pow(2).mean() + 1e-6

        snr_db = random.uniform(5, 20)
        snr = 10 ** (snr_db / 10)

        scale = torch.sqrt(signal_power / (snr * noise_power))
        return audio + scale * noise

    def __len__(self):
        return len(self.song_list)

    def __getitem__(self, idx):
        song_path, label = self.song_list[idx]

        drums = self.load_audio(os.path.join(song_path, "drums.wav"))
        vocals = self.load_audio(os.path.join(song_path, "vocals.wav"))
        bass = self.load_audio(os.path.join(song_path, "bass.wav"))
        other = self.load_audio(os.path.join(song_path, "other.wav"))

        if self.train:
            drums *= random.uniform(0.6, 1.4)
            vocals *= random.uniform(0.6, 1.4)
            bass *= random.uniform(0.6, 1.4)
            other *= random.uniform(0.6, 1.4)

        mixed = (drums + vocals + bass + other) / 4.0
        mixed = torch.clamp(mixed, -1.0, 1.0)

        if self.train and random.random() < 0.8:
            mixed = self.add_noise(mixed)

        # MEL
        mel = self.mel_transform(mixed)
        mel = self.db_transform(mel)

        # Normalize
        mel = (mel - mel.mean()) / (mel.std() + 1e-6)

        if mel.shape[1] > self.target_frames:
            mel = mel[:, :self.target_frames]
        else:
            pad_width = self.target_frames - mel.shape[1]
            mel = torch.nn.functional.pad(mel, (0, pad_width))

        if self.train:
            if random.random() < 0.5:
                mel = self.time_mask(mel)
            if random.random() < 0.5:
                mel = self.freq_mask(mel)

        return mel, label

In [9]:
dataset_test = MashupDataset(train_songs)
sample_mel, sample_label = dataset_test[0]

print("Log-Mel shape:", sample_mel.shape)
print("Label:", sample_label)

Log-Mel shape: torch.Size([128, 1024])
Label: 5


In [10]:
train_dataset = MashupDataset(train_songs, train=True)
val_dataset = MashupDataset(val_songs, train=False)

train_loader = DataLoader(train_dataset, batch_size=CFG.BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=CFG.BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

In [11]:
# # =========================================
# # CNN MODEL (MFCC INPUT)
# # =========================================

# class CNN_MFCC(nn.Module):
#     def __init__(self, num_classes=CFG.NUM_CLASSES):
#         super(CNN_MFCC, self).__init__()
        
#         self.features = nn.Sequential(
#             nn.Conv2d(1, 16, kernel_size=3, padding=1),
#             nn.BatchNorm2d(16),
#             nn.ReLU(),
#             nn.MaxPool2d(2),
            
#             nn.Conv2d(16, 32, kernel_size=3, padding=1),
#             nn.BatchNorm2d(32),
#             nn.ReLU(),
#             nn.MaxPool2d(2),
            
#             nn.Conv2d(32, 64, kernel_size=3, padding=1),
#             nn.BatchNorm2d(64),
#             nn.ReLU(),
#             nn.MaxPool2d(2),
#         )
        
#         # We will compute this dynamically
#         self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        
#         self.classifier = nn.Sequential(
#             nn.Flatten(),
#             nn.Linear(64, 128),
#             nn.ReLU(),
#             nn.Dropout(0.3),
#             nn.Linear(128, num_classes)
#         )
        
#     def forward(self, x):
#         x = self.features(x)
#         x = self.global_pool(x)
#         x = self.classifier(x)
#         return x


# # Initialize model
# model = CNN_MFCC().to(device)

# print(model)

In [12]:
# # =========================================
# # LSTM MODEL (MFCC SEQUENCE)
# # =========================================

# class LSTM_MFCC(nn.Module):
#     def __init__(self, input_size=CFG.N_MFCC, hidden_size=128, num_layers=2, num_classes=CFG.NUM_CLASSES):
#         super(LSTM_MFCC, self).__init__()
        
#         self.lstm = nn.LSTM(
#             input_size=input_size,
#             hidden_size=hidden_size,
#             num_layers=num_layers,
#             batch_first=True,
#             bidirectional=True
#         )
        
#         self.dropout = nn.Dropout(0.3)
        
#         self.fc = nn.Linear(hidden_size * 2, num_classes)
        
#     def forward(self, x):
#         # x shape: (batch, 1, 40, T)
        
#         x = x.squeeze(1)          # (batch, 40, T)
#         x = x.permute(0, 2, 1)    # (batch, T, 40)
        
#         output, _ = self.lstm(x)
        
#         # Take last time step
#         last_output = output[:, -1, :]
        
#         out = self.dropout(last_output)
#         out = self.fc(out)
        
#         return out

In [13]:
# model = LSTM_MFCC().to(device)
# print(model)

In [14]:
model = ASTForAudioClassification.from_pretrained(
    "MIT/ast-finetuned-audioset-10-10-0.4593",
    num_labels=CFG.NUM_CLASSES,
    ignore_mismatched_sizes=True
)
if torch.cuda.device_count() > 1:
    print("Using", torch.cuda.device_count(), "GPUs")
    model = nn.DataParallel(model)

model = model.to(device)

print("Model ready.")

Some weights of ASTForAudioClassification were not initialized from the model checkpoint at MIT/ast-finetuned-audioset-10-10-0.4593 and are newly initialized because the shapes did not match:
- classifier.dense.bias: found shape torch.Size([527]) in the checkpoint and torch.Size([10]) in the model instantiated
- classifier.dense.weight: found shape torch.Size([527, 768]) in the checkpoint and torch.Size([10, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Using 2 GPUs
Model ready.


In [15]:
# =========================================
# Freeze Early Transformer Layers (Stability)
# =========================================

for name, param in model.named_parameters():
    if "encoder.layer.0" in name or "encoder.layer.1" in name:
        param.requires_grad = False

print("First 2 transformer layers frozen.")

First 2 transformer layers frozen.


In [16]:
# from kaggle_secrets import UserSecretsClient
# user_secrets = UserSecretsClient()
# secret_value_0 = user_secrets.get_secret("DLGENAI_WANDB_API_KEY")
# os.environ["WANDB_API_KEY"] = secret_value_0
# wandb.login()

In [17]:
# =========================================
# AST v2 OPTIMIZER SETUP
# =========================================

from transformers import get_cosine_schedule_with_warmup
from torch.cuda.amp import GradScaler

criterion = nn.CrossEntropyLoss(label_smoothing=CFG.LABEL_SMOOTHING)

optimizer = torch.optim.AdamW(
    model.module.parameters() if isinstance(model, nn.DataParallel)
    else model.parameters(),
    lr=CFG.LR,
    weight_decay=CFG.WEIGHT_DECAY
)

total_steps = len(train_loader) * CFG.EPOCHS
warmup_steps = int(0.1 * total_steps)

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

scaler = GradScaler()

print("AST v2 optimizer ready.")

AST v2 optimizer ready.


In [18]:
# import wandb

# if CFG.TRAIN_MODE:
#     wandb.init(
#         project="24f2001786-t12026",
#         name="ast_V3_model",
#         config={
#             "epochs": CFG.EPOCHS,
#             "batch_size": CFG.BATCH_SIZE,
#             "lr": CFG.LR,
#             "n_mfcc": CFG.N_MELS
#         },
#         reinit=True
#     )

# print("WandB safely initialized.")

In [ ]:
# =========================================
# ADVANCED TRAINING LOOP (AMP + COSINE + MIXUP + UNFREEZE)
# =========================================

best_f1 = 0.0
UNFREEZE_EPOCH = 4


# -----------------------------------------
# MIXUP FUNCTION
# -----------------------------------------
def mixup_data(x, y, alpha=0.4):
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1

    batch_size = x.size()[0]
    index = torch.randperm(batch_size).to(x.device)

    mixed_x = lam * x + (1 - lam) * x[index]
    y_a, y_b = y, y[index]

    return mixed_x, y_a, y_b, lam


# -----------------------------------------
# TRAIN FUNCTION
# -----------------------------------------
def train_one_epoch(model, loader):
    model.train()
    total_loss = 0
    all_preds = []
    all_labels = []

    loop = tqdm(loader, desc="Training", leave=False)

    for inputs, labels in loop:

        inputs = inputs.to(device)
        inputs = inputs.permute(0, 2, 1)  # (B, T, M)
        labels = labels.to(device)

        optimizer.zero_grad()

        # 50% chance of mixup
        if np.random.rand() < 0.5:
            inputs, targets_a, targets_b, lam = mixup_data(inputs, labels)

            with autocast():
                outputs = model(inputs).logits
                loss = lam * criterion(outputs, targets_a) + \
                       (1 - lam) * criterion(outputs, targets_b)
        else:
            with autocast():
                outputs = model(inputs).logits
                loss = criterion(outputs, labels)

        scaler.scale(loss).backward()

        # Gradient clipping
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        scaler.step(optimizer)
        scaler.update()

        scheduler.step()

        total_loss += loss.item()

        preds = torch.argmax(outputs, dim=1)
        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())

        loop.set_postfix(loss=loss.item())

    epoch_loss = total_loss / len(loader)
    epoch_f1 = f1_score(all_labels, all_preds, average="macro")
    epoch_acc = accuracy_score(all_labels, all_preds)

    return epoch_loss, epoch_f1, epoch_acc


# -----------------------------------------
# VALIDATION FUNCTION
# -----------------------------------------
def validate(model, loader):
    model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []

    loop = tqdm(loader, desc="Validation", leave=False)

    with torch.no_grad():
        for inputs, labels in loop:

            inputs = inputs.to(device)
            inputs = inputs.permute(0, 2, 1)
            labels = labels.to(device)

            with autocast():
                outputs = model(inputs).logits
                loss = criterion(outputs, labels)

            total_loss += loss.item()

            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.detach().cpu().numpy())
            all_labels.extend(labels.detach().cpu().numpy())

            loop.set_postfix(loss=loss.item())

    epoch_loss = total_loss / len(loader)
    epoch_f1 = f1_score(all_labels, all_preds, average="macro")
    epoch_acc = accuracy_score(all_labels, all_preds)

    return epoch_loss, epoch_f1, epoch_acc


# =========================================
# TRAINING START
# =========================================

if CFG.TRAIN_MODE:

    for epoch in range(CFG.EPOCHS):

        print(f"\nEpoch {epoch+1}/{CFG.EPOCHS}")

        # 🔓 Unfreeze all layers after few epochs
        if epoch == UNFREEZE_EPOCH:
            print("🔓 Unfreezing all transformer layers...")
            for param in model.parameters():
                param.requires_grad = True

        train_loss, train_f1, train_acc = train_one_epoch(model, train_loader)
        val_loss, val_f1, val_acc = validate(model, val_loader)

        print(f"Train Loss: {train_loss:.4f} | Train F1: {train_f1:.4f} | Train Acc: {train_acc:.4f}")
        print(f"Val   Loss: {val_loss:.4f} | Val   F1: {val_f1:.4f} | Val   Acc: {val_acc:.4f}")

        # wandb.log({
        #     "epoch": epoch + 1,
        #     "train_loss": train_loss,
        #     "train_f1": train_f1,
        #     "train_acc": train_acc,
        #     "val_loss": val_loss,
        #     "val_f1": val_f1,
        #     "val_acc": val_acc,
        #     "lr": scheduler.get_last_lr()[0]
        # })

        if val_f1 > best_f1:
            best_f1 = val_f1

            torch.save(
                model.module.state_dict() if isinstance(model, nn.DataParallel)
                else model.state_dict(),
                CFG.MODEL_SAVE_PATH
            )

            print("✅ Best model saved!")

In [18]:
# Create fresh base model
base_model = ASTForAudioClassification.from_pretrained(
    "MIT/ast-finetuned-audioset-10-10-0.4593",
    num_labels=CFG.NUM_CLASSES,
    ignore_mismatched_sizes=True
)

MODEL_PATH = "/kaggle/input/models/genrede/ast-4/pytorch/default/1/best_ast_v4.pth"

state_dict = torch.load(MODEL_PATH, map_location="cpu")
base_model.load_state_dict(state_dict)

base_model = base_model.to(device)

# Wrap AFTER loading
if torch.cuda.device_count() > 1:
    print("Using", torch.cuda.device_count(), "GPUs")
    model = nn.DataParallel(base_model)
else:
    model = base_model

model.eval()

print("Model loaded successfully.")

Some weights of ASTForAudioClassification were not initialized from the model checkpoint at MIT/ast-finetuned-audioset-10-10-0.4593 and are newly initialized because the shapes did not match:
- classifier.dense.bias: found shape torch.Size([527]) in the checkpoint and torch.Size([10]) in the model instantiated
- classifier.dense.weight: found shape torch.Size([527, 768]) in the checkpoint and torch.Size([10, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Using 2 GPUs
Model loaded successfully.


In [19]:
# =========================================
# MULTI-CROP TEST DATASET (TORCHAUDIO)
# =========================================

class TestDataset(Dataset):
    def __init__(self, test_csv_path, data_root, num_crops=5):
        self.test_df = pd.read_csv(test_csv_path)
        self.data_root = data_root
        self.sample_rate = CFG.SAMPLE_RATE
        self.chunk_length = CFG.MAX_LENGTH
        self.num_crops = num_crops
        
        self.mel_transform = T.MelSpectrogram(
            sample_rate=self.sample_rate,
            n_fft=1024,
            hop_length=320,
            n_mels=CFG.N_MELS,
            f_max=8000
        )
        
        self.db_transform = T.AmplitudeToDB()

    def __len__(self):
        return len(self.test_df)

    def __getitem__(self, idx):
        row = self.test_df.iloc[idx]
        file_id = row["id"]
        filename = row["filename"]

        path = os.path.join(CFG.DATA_ROOT, filename)
        waveform, sr = torchaudio.load(path)

        if sr != self.sample_rate:
            waveform = torchaudio.functional.resample(
                waveform, sr, self.sample_rate
            )

        waveform = waveform.mean(dim=0)

        total_len = waveform.shape[0]
        crops = []

        if total_len <= self.chunk_length:
            waveform = torch.nn.functional.pad(
                waveform, (0, self.chunk_length - total_len)
            )
            starts = [0] * self.num_crops
        else:
            stride = (total_len - self.chunk_length) // (self.num_crops - 1)
            starts = [i * stride for i in range(self.num_crops)]

        for start in starts:
            chunk = waveform[start:start + self.chunk_length]
            mel = self.mel_transform(chunk)
            mel = self.db_transform(mel)
            mel = (mel - mel.mean()) / (mel.std() + 1e-6)
            
            # 🔥 FORCE EXACT 1024 FRAMES
            if mel.shape[1] > CFG.TARGET_FRAMES:
                mel = mel[:, :CFG.TARGET_FRAMES]
            else:
                pad_width = CFG.TARGET_FRAMES - mel.shape[1]
                mel = torch.nn.functional.pad(mel, (0, pad_width))
            
            crops.append(mel)

        crops = torch.stack(crops)  # (num_crops, 128, 1024)

        return crops, file_id

In [20]:
# =========================================
# MULTI-CROP INFERENCE
# =========================================

test_dataset = TestDataset(CFG.TEST_CSV, CFG.DATA_ROOT, num_crops=5)

test_loader = DataLoader(
    test_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

predictions = []
ids = []

model.eval()

with torch.no_grad():
    for crops, file_ids in tqdm(test_loader):

        crops = crops.to(device)  # (B, C, 128, 1024)

        B, C, M, T = crops.shape
        crops = crops.view(B * C, M, T)
        crops = crops.permute(0, 2, 1)  # (B*C, T, M)

        with autocast():
            outputs = model(crops).logits  # (B*C, 10)

        outputs = outputs.view(B, C, -1)
        outputs = outputs.mean(dim=1)  # average logits

        preds = torch.argmax(outputs, dim=1)

        predictions.extend(preds.cpu().numpy())
        ids.extend([int(i) for i in file_ids])

predicted_genres = [idx2label[p] for p in predictions]

submission = pd.DataFrame({
    "id": ids,
    "genre": predicted_genres
})

submission.to_csv("/kaggle/working/submission.csv", index=False)

print("Submission file created.")

100%|██████████| 755/755 [03:15<00:00,  3.87it/s]

Submission file created.
